# DKC2 HD-Pack: SD 1.5 + ControlNet Tile + IP-Adapter

**KI-Neuzeichnung von DKC2 Tiles — Stil via Referenzbilder steuerbar (kein Training nötig)**

```
DKC2 Viewer Export-ZIP  +  [optionale Referenzbilder]  →  SD-Pipeline  →  HD-ZIP
```

### Drei Qualitätsstufen

| Stufe | Was | Vorteil | Wann |
|-------|-----|---------|------|
| **1 — Basis** | SD + ControlNet, nur Prompt | Sofort startbereit | Erste Tests |
| **2 — IP-Adapter** | + Rareware/DKC-Render als Referenz | Stil ohne Training übertragen | Wenn du Referenzbilder hast |
| **3 — LoRA** | + trainiertes LoRA | Stärkste Stil-Kontrolle | Nach ~1h Training |

### Ablauf
1. **Zelle 1** — Libraries installieren (~3 min)
2. **Zelle 2** — SD 1.5 + ControlNet laden
3. **Zelle 3** — IP-Adapter laden (optional, für Referenzbilder)
4. **Zelle 4** — Einstellungen
5. **Zelle 5** — Referenzbild hochladen (optional, für IP-Adapter)
6. **Zelle 6** — Export-ZIP verarbeiten
7. **Zelle 7** — HD-ZIP herunterladen
8. **Zellen 8-9** — LoRA Training (optional)

---
**Voraussetzung:** T4 GPU aktiv → `Bearbeiten → Notebook-Einstellungen → T4 GPU`

In [ ]:
# ZELLE 1: Libraries installieren (~3 Minuten)

!pip install diffusers==0.27.2 transformers accelerate xformers peft -q

import torch, sys

if torch.cuda.is_available():
    gpu  = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✓ GPU: {gpu}')
    print(f'✓ VRAM: {vram:.1f} GB')
else:
    print('FEHLER: Keine GPU! T4 aktivieren: Bearbeiten → Notebook-Einstellungen → T4 GPU')
    sys.exit(1)

print('Bereit.')

In [ ]:
# ZELLE 2: SD 1.5 + ControlNet Tile laden (~3 GB Download, ~2 Minuten)

from diffusers import StableDiffusionControlNetImg2ImgPipeline, ControlNetModel, UniPCMultistepScheduler
import torch

print('Lade ControlNet Tile (~1.5 GB)...')
controlnet = ControlNetModel.from_pretrained(
    'lllyasviel/control_v11f1e_sd15_tile',
    torch_dtype=torch.float16
)

print('Lade Stable Diffusion 1.5 (~2 GB)...')
pipe = StableDiffusionControlNetImg2ImgPipeline.from_pretrained(
    'runwayml/stable-diffusion-v1-5',
    controlnet=controlnet,
    torch_dtype=torch.float16,
    safety_checker=None,
    requires_safety_checker=False
)
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to('cuda')

try:
    pipe.enable_xformers_memory_efficient_attention()
    print('✓ xformers aktiv')
except:
    print('xformers nicht verfügbar — standard attention')

ip_adapter_loaded = False
lora_loaded       = False
style_ref_image   = None

print(f'✓ Pipeline bereit  ({torch.cuda.memory_allocated()/1e9:.1f} GB VRAM)')

In [ ]:
# ZELLE 3: IP-Adapter laden (optional — für Referenzbilder wie Rareware-Renders)
#
# IP-Adapter = "Bild-Prompt" — du zeigst dem Modell ein Referenzbild (Stil-Vorlage)
# und es übernimmt Farben, Texturen und Zeichenstil daraus. Kein Training nötig.
#
# Wenn du keine Referenzbilder hast: diese Zelle ÜBERSPRINGEN.

print('Lade IP-Adapter (~300 MB)...')
pipe.load_ip_adapter(
    'h94/IP-Adapter',
    subfolder='models',
    weight_name='ip-adapter_sd15.bin'
)
ip_adapter_loaded = True
print('✓ IP-Adapter geladen!')
print()
print('Jetzt Zelle 5 ausführen → Referenzbild hochladen (z.B. Rareware Render PNG)')

In [ ]:
# ZELLE 4: Einstellungen
# ──────────────────────────────────────────────────────────────────────────────
#
# DENOISE_STRENGTH: Wie stark SD das Bild verändert
#   0.30 = kaum Änderung (sicher, aber wenig Detail-Gewinn)
#   0.55 = gute Balance ← Standard
#   0.75 = viel KI-Kreativität (kann Proportionen leicht verändern)
#
# CONTROLNET_SCALE: Wie streng die Originalstruktur erhalten bleibt
#   0.50 = locker (SD hat viel Freiheit bei Formen)
#   0.80 = strukturtreu ← Standard
#   0.95 = sehr eng (fast identische Umrisse wie Original)
#
# IP_ADAPTER_SCALE: Wie stark das Referenzbild den Stil bestimmt
#   0.30 = dezenter Stil-Einfluss
#   0.60 = deutlicher Stil-Einfluss ← Standard
#   0.90 = sehr starker Stil-Einfluss (Referenzbild dominiert alles)
#
# OUTPUT_SCALE: Skalierungsfaktor für die Ausgabe
#   4 = 384×384px pro Tile (schnell, ~2-3s/Tile auf T4) ← Standard
#   6 = 576×576px pro Tile (schärfer, ~4-5s/Tile, mehr VRAM)
# ──────────────────────────────────────────────────────────────────────────────

POSITIVE_PROMPT = (
    'high quality retro game art, cartoon style, vibrant saturated colors, '
    'detailed texture, donkey kong country 2, tropical jungle pirate theme, '
    'clean sharp linework, professional game asset, SNES art upscale'
)
NEGATIVE_PROMPT = (
    'blurry, photorealistic, 3d render, low quality, noisy, distorted, '
    'watermark, text, human face, extra limbs'
)

DENOISE_STRENGTH  = 0.55
CONTROLNET_SCALE  = 0.80
IP_ADAPTER_SCALE  = 0.60   # nur wirksam wenn IP-Adapter geladen (Zelle 3)

OUTPUT_SCALE      = 4      # 4 oder 6
OUTPUT_SIZE       = 96 * OUTPUT_SCALE
SD_MAX_SIZE       = 640    # Tiles größer als das → bicubic fallback (verhindert OOM)

NUM_STEPS         = 20
GUIDANCE_SCALE    = 7.5
SEED              = 42

# IP-Adapter Skala setzen wenn geladen
if ip_adapter_loaded:
    pipe.set_ip_adapter_scale(IP_ADAPTER_SCALE)

print('── Einstellungen ──────────────────────────────────')
print(f'  Output:           {OUTPUT_SIZE}×{OUTPUT_SIZE}px  (scaleFactor={OUTPUT_SCALE})')
print(f'  DENOISE_STRENGTH: {DENOISE_STRENGTH}')
print(f'  CONTROLNET_SCALE: {CONTROLNET_SCALE}')
print(f'  IP_ADAPTER_SCALE: {IP_ADAPTER_SCALE}  (aktiv: {ip_adapter_loaded})')
print(f'  LoRA aktiv:       {lora_loaded}')
print(f'  Referenzbild:     {"geladen" if style_ref_image is not None else "keines"}')
print(f'  Steps / CFG:      {NUM_STEPS} / {GUIDANCE_SCALE}')

In [ ]:
# ZELLE 5: Referenzbild hochladen (nur wenn IP-Adapter geladen — Zelle 3)
#
# Geeignete Referenzbilder für DKC2-Stil:
#   ✅ Rareware Render-Grafiken (DKC2 Promo-Art, Charakterrender aus den 90ern)
#   ✅ DKC: Tropical Freeze / DKC Returns HD Screenshots
#   ✅ DKC2 Concept Art
#   ✅ Eigene Zeichnungen im gewünschten Stil
#
# Hinweise:
#   - Mehrere Bilder möglich — das Modell mittelt den Stil
#   - Auflösung egal (wird intern auf 224×224 skaliert)
#   - IP_ADAPTER_SCALE in Zelle 4 steuert wie stark der Stil übernommen wird
#   - Zelle überspringen wenn kein Referenzbild gewünscht

from google.colab import files as colab_files
from PIL import Image
import io

if not ip_adapter_loaded:
    print('⚠️  IP-Adapter nicht geladen — Zelle 3 zuerst ausführen!')
else:
    print('Referenzbild(er) hochladen (PNG/JPG):')
    up = colab_files.upload()

    if len(up) == 1:
        # Einzelnes Bild
        fname = list(up.keys())[0]
        style_ref_image = Image.open(io.BytesIO(list(up.values())[0])).convert('RGB')
        print(f'✓ Referenzbild: {fname}  ({style_ref_image.size[0]}×{style_ref_image.size[1]}px)')
    elif len(up) > 1:
        # Mehrere Bilder → zu einer Collage zusammenfügen (IP-Adapter mittelt Stile)
        imgs = [Image.open(io.BytesIO(d)).convert('RGB').resize((256, 256), Image.LANCZOS)
                for d in up.values()]
        cols = min(len(imgs), 4)
        rows = (len(imgs) + cols - 1) // cols
        collage = Image.new('RGB', (cols * 256, rows * 256))
        for i, img in enumerate(imgs):
            collage.paste(img, ((i % cols) * 256, (i // cols) * 256))
        style_ref_image = collage
        print(f'✓ {len(imgs)} Referenzbilder als {cols}×{rows}-Collage kombiniert')
    else:
        print('Kein Bild hochgeladen — weiter ohne Referenz')
        style_ref_image = None

    if style_ref_image is not None:
        print(f'  IP-Adapter Skala: {IP_ADAPTER_SCALE}  (anpassen in Zelle 4 → IP_ADAPTER_SCALE)')
        print('  Tipp: 0.6 = guter Balance-Punkt zwischen Original-Stil und Referenz-Stil')

In [ ]:
# ZELLE 6: Export-ZIP hochladen + alle Tiles verarbeiten

import zipfile, io, json, time
import numpy as np
import torch
from PIL import Image
from google.colab import files as colab_files

print('DKC2 Export-ZIP hochladen (aus Viewer: Catalog View → Export ZIP):')
up = colab_files.upload()
zip_name = list(up.keys())[0]

in_zip      = zipfile.ZipFile(io.BytesIO(up[zip_name]))
all_names   = [n for n in in_zip.namelist() if not n.endswith('/')]
png_names   = [n for n in all_names if n.lower().endswith('.png')]
other_names = [n for n in all_names if not n.lower().endswith('.png')]

mode_str = ''
if ip_adapter_loaded and style_ref_image is not None:
    mode_str = 'SD + ControlNet + IP-Adapter (Referenzbild)'
elif lora_loaded:
    mode_str = 'SD + ControlNet + LoRA'
else:
    mode_str = 'SD + ControlNet (nur Prompt)'

print(f'ZIP: {zip_name}')
print(f'  PNG-Dateien:  {len(png_names)}')
print(f'  Modus:        {mode_str}')
print(f'  Output:       {OUTPUT_SIZE}×{OUTPUT_SIZE}px ({OUTPUT_SCALE}x)')
print()

def process_tile(img_bytes):
    """Verarbeitet ein Tile durch SD+ControlNet (+optional IP-Adapter)."""
    img      = Image.open(io.BytesIO(img_bytes))
    has_alpha = (img.mode == 'RGBA')
    alpha    = img.split()[3] if has_alpha else None
    rgb      = img.convert('RGB')
    orig_w, orig_h = rgb.size

    # Zielgröße — auf 64er-Raster für SD
    target_w = orig_w * OUTPUT_SCALE
    target_h = orig_h * OUTPUT_SCALE
    sd_w = max(64, (target_w // 64) * 64)
    sd_h = max(64, (target_h // 64) * 64)

    rgb_up = rgb.resize((sd_w, sd_h), Image.BICUBIC)

    if sd_w > SD_MAX_SIZE or sd_h > SD_MAX_SIZE:
        # Zu groß für SD → nur bicubisch
        result_rgb = rgb.resize((target_w, target_h), Image.BICUBIC)
        method = 'bicubic'
    else:
        gen = torch.Generator('cuda').manual_seed(SEED)

        pipe_kwargs = dict(
            prompt=POSITIVE_PROMPT,
            negative_prompt=NEGATIVE_PROMPT,
            image=rgb_up,
            control_image=rgb_up,
            num_inference_steps=NUM_STEPS,
            guidance_scale=GUIDANCE_SCALE,
            strength=DENOISE_STRENGTH,
            controlnet_conditioning_scale=CONTROLNET_SCALE,
            generator=gen,
        )

        # IP-Adapter Referenzbild hinzufügen wenn vorhanden
        if ip_adapter_loaded and style_ref_image is not None:
            pipe_kwargs['ip_adapter_image'] = style_ref_image

        out = pipe(**pipe_kwargs).images[0]
        result_rgb = out.resize((target_w, target_h), Image.LANCZOS) if out.size != (target_w, target_h) else out
        method = 'sd+ip' if (ip_adapter_loaded and style_ref_image is not None) else 'sd'

    if has_alpha and alpha is not None:
        alpha_up = alpha.resize((target_w, target_h), Image.NEAREST)
        result   = Image.merge('RGBA', (*result_rgb.split(), alpha_up))
    else:
        result = result_rgb

    buf = io.BytesIO()
    result.save(buf, 'PNG')
    return buf.getvalue(), method

# ── Verarbeitung ─────────────────────────────────────────────────────────────
out_buf  = io.BytesIO()
out_zip  = zipfile.ZipFile(out_buf, 'w', zipfile.ZIP_DEFLATED)
errors, sd_count, ip_count, bic_count = [], 0, 0, 0
t_start  = time.time()

for i, name in enumerate(png_names):
    t0 = time.time()
    try:
        data, method = process_tile(in_zip.read(name))
        out_zip.writestr(name, data)
        if method == 'sd+ip': ip_count += 1
        elif method == 'sd':  sd_count += 1
        else:                 bic_count += 1
        dt   = time.time() - t0
        done = i + 1
        eta  = (time.time() - t_start) / done * (len(png_names) - done)
        icons = {'sd+ip': '🎨✨', 'sd': '🎨', 'bicubic': '📐'}
        print(f'[{done:3d}/{len(png_names)}] {icons.get(method,"?")} {name.split("/")[-1]:36s} {dt:.1f}s  ETA {eta:.0f}s')
    except Exception as e:
        out_zip.writestr(name, in_zip.read(name))
        errors.append(name)
        print(f'[{i+1:3d}/{len(png_names)}] ❌ {name.split("/")[-1]}: {e}')

for name in other_names:
    data = in_zip.read(name)
    if name.endswith('manifest.json'):
        try:
            m = json.loads(data)
            m['scaleFactor'] = OUTPUT_SCALE
            m['upscaleModel'] = mode_str
            if lora_loaded: m['loraModel'] = 'dkc2_lora'
            data = json.dumps(m, indent=2).encode()
        except Exception as e:
            print(f'manifest update fehler: {e}')
    out_zip.writestr(name, data)

out_zip.close()
total = time.time() - t_start
ok    = len(png_names) - len(errors)

print(f'\n══════════════════════════════════════════')
print(f'Fertig: {ok}/{len(png_names)} Tiles in {total:.0f}s')
print(f'  🎨✨ SD + IP-Adapter: {ip_count}')
print(f'  🎨   SD only:         {sd_count}')
print(f'  📐   Bicubic:         {bic_count}')
if errors: print(f'  ❌   Fehler:          {len(errors)}')

In [ ]:
# ZELLE 7: HD-ZIP herunterladen

from google.colab import files as colab_files

tag      = f'_ip{IP_ADAPTER_SCALE}' if (ip_adapter_loaded and style_ref_image is not None) else ''
tag     += '_lora' if lora_loaded else ''
out_name = zip_name.replace('.zip', f'_sd{tag}_d{DENOISE_STRENGTH}_c{CONTROLNET_SCALE}_{OUTPUT_SCALE}x.zip')

with open(out_name, 'wb') as f:
    f.write(out_buf.getvalue())

size_mb = len(out_buf.getvalue()) / 1e6
print(f'Download: {out_name}  ({size_mb:.1f} MB)')
colab_files.download(out_name)
print()
print('Im DKC2 Viewer: Catalog View → Import HD → diese ZIP laden → HD-Toggle aktivieren')

---
## LoRA Training (optional — für stärksten Stil-Effekt)

LoRA lernt den Stil aus deinen Referenzbildern und bäckt ihn fest ins Modell.
IP-Adapter ist flexibler (Bild wechseln ohne neu trainieren), LoRA ist stärker.

**Benötigt:** ~20-50 Screenshots DKC: Tropical Freeze oder DKC Returns HD  
**Zeit:** ~1-2 Stunden auf T4

In [ ]:
# ZELLE 8: LoRA Training-Bilder hochladen

import os, io
from PIL import Image
from google.colab import files as colab_files

TRAINING_DIR    = '/content/dkc_training_images'
INSTANCE_PROMPT = 'dkc2style donkey kong country jungle pirate game art'
os.makedirs(TRAINING_DIR, exist_ok=True)

print('Training-Bilder hochladen (DKC:TF / DKC Returns HD Screenshots):')
up = colab_files.upload()

for fname, data in up.items():
    img = Image.open(io.BytesIO(data)).convert('RGB').resize((512, 512), Image.LANCZOS)
    dest = os.path.join(TRAINING_DIR, os.path.splitext(fname)[0] + '.png')
    img.save(dest, 'PNG')
    print(f'  {fname} → 512×512')

n = len([f for f in os.listdir(TRAINING_DIR) if f.endswith('.png')])
LORA_STEPS = max(500, min(2000, n * 40))
print(f'\nBilder gesamt: {n}  |  Geplante Schritte: {LORA_STEPS}  |  ~{LORA_STEPS*4//60} Minuten')

In [ ]:
# ZELLE 9: LoRA Training + laden

!pip install accelerate peft bitsandbytes -q
!git clone --depth=1 https://github.com/huggingface/diffusers /content/diffusers_train -q
!pip install -r /content/diffusers_train/examples/dreambooth/requirements.txt -q

LORA_OUT = '/content/dkc2_lora'
os.makedirs(LORA_OUT, exist_ok=True)

!accelerate launch /content/diffusers_train/examples/dreambooth/train_dreambooth_lora.py \
  --pretrained_model_name_or_path='runwayml/stable-diffusion-v1-5' \
  --instance_data_dir={TRAINING_DIR} \
  --output_dir={LORA_OUT} \
  --instance_prompt='{INSTANCE_PROMPT}' \
  --resolution=512 \
  --train_batch_size=1 \
  --gradient_accumulation_steps=2 \
  --learning_rate=1e-4 \
  --lr_scheduler='cosine_with_restarts' \
  --lr_warmup_steps=50 \
  --max_train_steps={LORA_STEPS} \
  --mixed_precision='fp16' \
  --enable_xformers_memory_efficient_attention \
  --seed={SEED}

lora_file = os.path.join(LORA_OUT, 'pytorch_lora_weights.safetensors')
if os.path.exists(lora_file):
    pipe.load_lora_weights(LORA_OUT)
    lora_loaded = True
    POSITIVE_PROMPT = INSTANCE_PROMPT + ', ' + POSITIVE_PROMPT
    print(f'\n✓ LoRA geladen ({os.path.getsize(lora_file)/1e6:.1f} MB)')
    print('Jetzt Zellen 6+7 erneut ausführen → HD-ZIP mit LoRA-Stil')
    from google.colab import files as colab_files
    colab_files.download(lora_file)
else:
    print('⚠️  Training fehlgeschlagen — lora file not found')

---
## Kurzreferenz: Alle Einstellungen

```
DENOISE_STRENGTH   Wie stark SD das Bild verändert
  0.30             kaum Änderung (sicher)
  0.55  ◀ default  gute Balance
  0.75             viel KI-Freiheit

CONTROLNET_SCALE   Wie eng SD die Originalstruktur einhält
  0.50             locker (freie Formen)
  0.80  ◀ default  strukturtreu
  0.95             sehr eng am Original

IP_ADAPTER_SCALE   Wie stark das Referenzbild den Stil bestimmt
  0.30             dezenter Einfluss
  0.60  ◀ default  deutlicher Stil
  0.90             Referenz dominiert

OUTPUT_SCALE       Skalierungsfaktor
  4     ◀ default  384×384px, schnell (~2s/Tile)
  6                576×576px, schärfer (~4s/Tile)

NUM_STEPS          Qualität / Geschwindigkeit
  15               schnell
  20    ◀ default
  30               langsam aber besser
```

## Workflow-Empfehlungen

**Erster Test (keine Referenzbilder):**
> Zellen 1 → 2 → 4 → 6 → 7

**Mit Rareware-Renders als Stil-Referenz:**
> Zellen 1 → 2 → 3 → 4 → 5 (Bild hochladen) → 6 → 7

**Mit trainiertem LoRA:**
> Zellen 1 → 2 → 4 → 8 → 9 (Training, 1-2h) → 6 → 7

**Maximum (IP-Adapter + LoRA kombiniert):**
> Alle Zellen in Reihenfolge